In [1]:
"""Task B: Baseline GRU Model for Time Series Forecasting"""

import torch
import torch.nn as nn

class GRUForecast(nn.Module):
    """
    Baseline GRU forecaster for multi-step time series prediction
    
    Architecture:
    - GRU layer(s) to encode temporal patterns
    - Linear head to project last hidden state to horizon steps
    """
    
    def __init__(self, input_dim, hidden_dim=64, horizon=24, out_dim=1, 
                 num_layers=2, dropout=0.2):
        """
        Args:
            input_dim: Number of input features (d)
            hidden_dim: GRU hidden dimension
            horizon: Forecast horizon (H)
            out_dim: Output dimension (1 for single-target)
            num_layers: Number of GRU layers
            dropout: Dropout rate (applied between layers if num_layers > 1)
        """
        super().__init__()
        
        self.hidden_dim = hidden_dim
        self.horizon = horizon
        self.out_dim = out_dim
        self.num_layers = num_layers
        
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, horizon * out_dim)
        )
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        """Xavier initialization for linear layers"""
        for name, param in self.head.named_parameters():
            if 'weight' in name:
                nn.init.xavier_uniform_(param)
            elif 'bias' in name:
                nn.init.zeros_(param)
    
    def forward(self, x):
        """
        Forward pass
        
        Args:
            x: Input tensor of shape (batch_size, lookback, input_dim)
        
        Returns:
            Predictions of shape (batch_size, horizon, out_dim)
        """
        # GRU encoding
        gru_out, hidden = self.gru(x)  # gru_out: (B, L, hidden)
        
        # Take last hidden state (or use all? paper uses last)
        if self.num_layers > 1:
            # For multi-layer: take last layer's last timestep
            last_hidden = gru_out[:, -1, :]  # (B, hidden)
        else:
            last_hidden = hidden[-1]  # (B, hidden)
        
        # Linear projection to horizon steps
        predictions = self.head(last_hidden)  # (B, H * out_dim)
        
        # Reshape to (B, H, out_dim)
        predictions = predictions.view(-1, self.horizon, self.out_dim)
        
        return predictions
    
    def get_params_count(self):
        """Return number of trainable parameters"""
        return sum(p.numel() for p in self.parameters())

class LSTMForecast(nn.Module):
    """Alternative baseline using LSTM instead of GRU"""
    
    def __init__(self, input_dim, hidden_dim=64, horizon=24, out_dim=1, 
                 num_layers=2, dropout=0.2):
        super().__init__()
        
        self.horizon = horizon
        self.out_dim = out_dim
        
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        
        self.head = nn.Linear(hidden_dim, horizon * out_dim)
    
    def forward(self, x):
        lstm_out, (hidden, cell) = self.lstm(x)
        last_hidden = hidden[-1] if self.lstm.num_layers > 1 else hidden
        predictions = self.head(last_hidden)
        return predictions.view(-1, self.horizon, self.out_dim)

# Test the model
if __name__ == "__main__":
    model = GRUForecast(input_dim=7, hidden_dim=64, horizon=24)
    x = torch.randn(32, 96, 7)  # (B, L, d)
    y = model(x)
    print(f"Input shape: {x.shape}")
    print(f"Output shape: {y.shape}")
    print(f"Parameters: {model.get_params_count():,}")

Input shape: torch.Size([32, 96, 7])
Output shape: torch.Size([32, 24, 1])
Parameters: 41,848
